In [1]:
import pdfplumber
import pandas as pd
import re

In [7]:
pdf_file = "Remittance_Cenco.pdf"

# 1️⃣ Extraer todo el texto
full_text = []
with pdfplumber.open(pdf_file) as pdf:
    for page in pdf.pages:
        txt = page.extract_text()
        if txt:
            full_text.append(txt)

full_text = "\n".join(full_text)

In [8]:
# 2️⃣ Filtrar solo líneas que parecen ser registros de la tabla
#    (las que comienzan con los códigos de movimiento)
patron_linea = re.compile(r'^(FS|DEV|LTG|CH|FPM)\b.*', re.MULTILINE)
lineas = patron_linea.findall(full_text)
# Ojo: findall con este patrón devuelve solo el grupo (FS/DEV/…), 
#      por eso mejor usar finditer:
lineas = [m.group(0) for m in re.finditer(patron_linea, full_text)]

# 3️⃣ Separar columnas
#    El PDF usa espacios grandes como separador, por eso usamos
#    dos o más espacios como delimitador.
data = [re.split(r'\s{2,}', linea.strip()) for linea in lineas]

# 4️⃣ Crear DataFrame
df = pd.DataFrame(data)

In [10]:
# 5️⃣ Limpieza básica: quitar espacios y renombrar columnas a mano
df = df.apply(lambda col: col.str.strip())
# Ajusta los nombres de columnas según tu necesidad:
df.columns = [f"col_{i}" for i in range(df.shape[1])]

df.head()

,col_0
0,FS FACTURA VENTA VPP2 2019435 ADM. JUMBO - SED...
1,DEV DEVOLUCION MERCANCIAAAA- PLAT - CROSSD DRO...
2,DEV DEVOLUCION MERCANCIAAAA- PLAT - CROSSD DRO...
3,DEV DEVOLUCION MERCANCIAAAA- PLAT - CROSS RANC...
4,DEV DEVOLUCION MERCANCIAAAA- PLAT - CROSS PERF...
